In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [14]:
no2 = pd.read_csv("./data/NO2_manual_1213m.csv")
wind_u = pd.read_csv("./data/WIND_U_manual_27929m.csv")
wind_v = pd.read_csv("./data/WIND_V_manual_27929m.csv")
temp = pd.read_csv("./data/TEMP_manual_27929m.csv")
co = pd.read_csv("./data/CO_manual_1213m.csv")

# Section 1: Data preprocessing 

first we establish the lagged dataset, which involves creating new features based on previous time steps of the data. For example, if we have a time series, we can create lagged features such as the n02 at time t-1, t-2, etc. This hopefully will allow us to capture the temporal dependencies in the data.

In [5]:
import requests


def fetch_elevations(batch, session):
    url = "https://api.open-elevation.com/api/v1/lookup"
    payload = {
        "locations": [
            {"latitude": latitude, "longitude": longitude} for latitude, longitude in batch
        ]
    }
    response = session.post(url, json=payload, timeout=60)
    response.raise_for_status()
    return response.json()["results"]


if "elevation" not in no2.columns:
    no2["elevation"] = np.nan

existing_elevation_lookup = (
    no2.loc[no2["elevation"].notna(), ["latitude", "longitude", "elevation"]]
    .drop_duplicates(subset=["latitude", "longitude"])
    .set_index(["latitude", "longitude"])["elevation"]
    .to_dict()
)

unique_coordinates = no2.loc[
    ~no2.set_index(["latitude", "longitude"]).index.isin(existing_elevation_lookup.keys()),
    ["latitude", "longitude"],
].drop_duplicates().copy()

session = requests.Session()
elevation_lookup = dict(existing_elevation_lookup)
batch_size = 500

for start in range(0, len(unique_coordinates), batch_size):
    batch_frame = unique_coordinates.iloc[start:start + batch_size][["latitude", "longitude"]]
    batch = list(batch_frame.itertuples(index=False, name=None))
    try:
        results = fetch_elevations(batch, session)
        for (latitude, longitude), result in zip(batch, results):
            elevation_lookup[(latitude, longitude)] = result["elevation"]
    except requests.RequestException as exc:
        print(f"Elevation lookup failed for batch starting at row {start}: {exc}")
        for latitude, longitude in batch:
            elevation_lookup[(latitude, longitude)] = np.nan

    no2["elevation"] = [
        elevation_lookup.get((latitude, longitude), np.nan)
        for latitude, longitude in no2[["latitude", "longitude"]].itertuples(index=False, name=None)
    ]
    no2.to_csv("./data/NO2_manual_1213m.csv", index=False)

    print(f"Processed batch {start // batch_size + 1} / {(len(unique_coordinates) + batch_size - 1) // batch_size} and checkpointed ./data/NO2_manual_1213m.csv.")
    print(no2[["latitude", "longitude", "elevation"]].head())

print(f"Filled elevation for {len(unique_coordinates)} previously missing unique latitude/longitude pairs.")

Filled elevation for 0 previously missing unique latitude/longitude pairs.


In [ ]:
import numpy as np
import pandas as pd

date_column = "timestamp"

no2[date_column] = pd.to_datetime(no2[date_column], errors="coerce")
no2 = no2.dropna(subset=[date_column]).copy()
no2["day"] = no2[date_column].dt.normalize()

value_column = "NO2"

daily_no2 = (
    no2.groupby(["latitude", "longitude", "day"], as_index=False)[value_column]
    .mean()
    .rename(columns={value_column: "no2"})
)

if "elevation" in no2.columns:
    elevation_by_location = (
        no2[["latitude", "longitude", "elevation"]]
        .dropna(subset=["latitude", "longitude"])
        .drop_duplicates(subset=["latitude", "longitude"])
    )
    daily_no2 = daily_no2.merge(elevation_by_location, on=["latitude", "longitude"], how="left")
else:
    daily_no2["elevation"] = np.nan

daily_no2 = daily_no2.sort_values(["latitude", "longitude", "day"]).reset_index(drop=True)

lag_columns = []
for lag in range(1, 8):
    column_name = f"no2_day{lag}"
    daily_no2[column_name] = daily_no2.groupby(["latitude", "longitude"])["no2"].shift(lag)
    lag_columns.append(column_name)

no2_lagged_df = daily_no2.rename(columns={"day": "datetime"})[
    ["latitude", "longitude", "elevation", "datetime"] + lag_columns
].dropna(subset=lag_columns).reset_index(drop=True)

print(f"Created lagged NO2 dataset with shape: {no2_lagged_df.shape}")
no2_lagged_df.head()

Created lagged NO2 dataset with shape: (1726422, 11)


,latitude,longitude,elevation,datetime,no2_day1,no2_day2,no2_day3,no2_day4,no2_day5,no2_day6,no2_day7
0,28.197678,77.151198,195.0,2019-01-10,0.000062,0.000065,0.000048,0.000052,0.000057,0.000042,0.000102
1,28.197678,77.151198,195.0,2019-01-11,0.000076,0.000062,0.000065,0.000048,0.000052,0.000057,0.000042
2,28.197678,77.151198,195.0,2019-01-12,0.000086,0.000076,0.000062,0.000065,0.000048,0.000052,0.000057
3,28.197678,77.151198,195.0,2019-01-14,0.000033,0.000086,0.000076,0.000062,0.000065,0.000048,0.000052
4,28.197678,77.151198,195.0,2019-01-15,0.000071,0.000033,0.000086,0.000076,0.000062,0.000065,0.000048


In [12]:
no2_lagged_df = no2_lagged_df.sort_values(["datetime", "latitude", "longitude"]).reset_index(drop=True)
no2_lagged_df.head()

,latitude,longitude,elevation,datetime,no2_day1,no2_day2,no2_day3,no2_day4,no2_day5,no2_day6,no2_day7
0,28.219475,77.151198,192.0,2019-01-09,0.000069,0.000062,0.000060,0.000052,0.000066,0.000039,0.000113
1,28.219475,77.162097,192.0,2019-01-09,0.000069,0.000119,0.000060,0.000056,0.000067,0.000036,0.000116
2,28.219475,77.172995,197.0,2019-01-09,0.000070,0.000124,0.000059,0.000056,0.000067,0.000036,0.000117
3,28.219475,77.183893,199.0,2019-01-09,0.000070,0.000122,0.000059,0.000056,0.000067,0.000036,0.000117
4,28.230373,77.151198,196.0,2019-01-09,0.000074,0.000099,0.000060,0.000054,0.000067,0.000044,0.000114


In [13]:
no2_lagged_df.to_csv("./data/no2_lagged.csv", index=False)

In [ ]:
timestamp_column = "timestamp"
if timestamp_column is None:
    raise KeyError("Could not find a timestamp/date column in no2.")

no2[timestamp_column] = pd.to_datetime(no2[timestamp_column], errors="coerce")
unique_dates = no2[timestamp_column].dt.date.dropna().unique()
print(len(unique_dates), "unique dates in the NO2 dataset.")

340 unique dates in the NO2 dataset.
